In [ ]:
import pandas as pd
df = pd.read_csv('./parking-regs.csv')
after_aug_2024 = df[df['IssueDate'] >= '2024-08-01']
before_aug_2024 = df[df['IssueDate'] < '2024-08-01']
num_v_a = after_aug_2024[after_aug_2024['Violation'] == 'NO STANDING-BUS STOP'].sum()['NumberViolations']
num_v_b = before_aug_2024[before_aug_2024['Violation'] == 'NO STANDING-BUS STOP'].sum()['NumberViolations']

correct = num_v_a + num_v_b
print(f"Total violations: {correct}")
print(f"Violations after Aug 2024: {num_v_a}")
print(f"Violations before Aug 2024: {num_v_b}")

Total violations: 433946
Violations after Aug 2024: 290778
Violations before Aug 2024: 143168


In [39]:
only_dec_2025 = df[(df['IssueDate'] >= '2025-12-01') & (df['IssueDate'] < '2026-01-01')]
only_dec_2025[only_dec_2025['Violation'] == 'MTA CAMERA VIOLATION - PARKING IN A BUS STOP'].sum()

IssueDate           2025-12-012025-12-012025-12-012025-12-012025-1...
Violation           MTA CAMERA VIOLATION - PARKING IN A BUS STOPMT...
FineAmount                                                      23250
NumberViolations                                                47984
TotalFineAmount                                               5052100
dtype: object

In [ ]:
import requests
from dotenv import load_dotenv
import os

load_dotenv(".env.local")
API_KEY_ID = os.environ["OPEN_DATA_API_KEY_ID"]
API_KEY_SECRET = os.environ["OPEN_DATA_API_KEY_SECRET"]

BASE_URL = "https://data.cityofnewyork.us/api/v3/views/nc67-uf89/query.json"

def run_query(q):
    resp = requests.get(BASE_URL, params={"query": q}, auth=(API_KEY_ID, API_KEY_SECRET))
    resp.raise_for_status()
    return resp.json()

# How many bus stop violations exist total?
print("MTA camera total:", len(run_query("SELECT `summons_number` WHERE `violation` = 'MTA CAMERA VIOLATION - PARKING IN A BUS STOP' LIMIT 50000")))
print("No standing total:", len(run_query("SELECT `summons_number` WHERE `violation` = 'NO STANDING-BUS STOP' LIMIT 50000")))

# Sample some to see what dates look like
print("\nSample MTA camera dates:")
for r in run_query("SELECT `issue_date`, `violation` WHERE `violation` = 'MTA CAMERA VIOLATION - PARKING IN A BUS STOP' LIMIT 5"):
    print(r.get('issue_date'), r.get('violation'))

In [13]:
output_path = "ticketing-data.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} rows to {output_path}")

Saved 86 rows to ticketing-data.csv


In [18]:
import requests
import pandas as pd
import time
from dotenv import load_dotenv
import os

load_dotenv(".env.local")
API_KEY_ID = os.environ["OPEN_DATA_API_KEY_ID"]
API_KEY_SECRET = os.environ["OPEN_DATA_API_KEY_SECRET"]

BASE_URL = "https://data.cityofnewyork.us/api/v3/views/nc67-uf89/query.json"
QUERY = """
SELECT
  `plate`, `state`, `license_type`, `summons_number`, `issue_date`,
  `violation_time`, `violation`, `judgment_entry_date`, `fine_amount`,
  `penalty_amount`, `interest_amount`, `reduction_amount`, `payment_amount`,
  `amount_due`, `precinct`, `county`, `issuing_agency`, `violation_status`,
  `summons_image`
WHERE
  `issue_date` >= '07/01/2024'
  AND (`violation` = 'MTA CAMERA VIOLATION - PARKING IN A BUS STOP'
    OR `violation` = 'NO STANDING-BUS STOP')
"""

all_rows = []
offset = 0
limit = 10000

while True:
    paginated_query = QUERY.strip() + f" LIMIT {limit} OFFSET {offset}"
    resp = requests.get(BASE_URL, params={"query": paginated_query}, auth=(API_KEY_ID, API_KEY_SECRET))
    resp.raise_for_status()
    rows = resp.json()

    if not rows:
        break

    all_rows.extend(rows)
    print(f"Fetched {len(all_rows):,} rows so far...")

    if len(rows) < limit:
        break

    offset += limit
    time.sleep(0.2)

df = pd.DataFrame(all_rows)
df = df[[c for c in df.columns if not c.startswith(":")]]
print(f"\nTotal rows: {len(df):,}")
df.head()



KeyboardInterrupt: 